# UQRoute-TC — saved APIBank and ToolAlpaca alignment coverage

**Scope:** The existing 24 clean development records from the Qwen 1.5B APIBank and ToolAlpaca format run. No inference or held-out cases.

**Output:** One compact per-case alignment audit in the saved pilot folder. Whole-output token evidence and meaningful-token eligibility are reported separately; parser status is not an official correctness label.


## Source revisions and saved records


In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib
import importlib.util
import json
import subprocess
import sys
from collections import Counter

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/UQRoute-TC')
output_dir = root / 'pilots/qwen25-15b-xml-json-dev-24-v1'
manifest_path = output_dir / 'manifest.json'
assert manifest_path.is_file(), 'Saved 24-case run is missing'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
benchmark_revision = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
code_revision = '73f541c857542b3d50cf9df1c7969b06b0644381'
assert manifest['benchmark_revision'] == benchmark_revision
assert manifest['model_id'] == 'Qwen/Qwen2.5-1.5B-Instruct'
assert manifest['model_revision'] == '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
assert manifest['mode'] == 'single_prompt_tool_calling'
case_ids = manifest['case_ids']
assert len(case_ids) == len(set(case_ids)) == 24

benchmark = Path('/content/robustbench-tc-release')
project = Path('/content/uqroute-tc-alignment-audit-v1')
for path, url, revision in (
    (benchmark, 'https://github.com/WillChow66/robustbench-tc-release.git', benchmark_revision),
    (project, 'https://github.com/RavindraSSK/uqroute-tc.git', code_revision),
):
    if not path.exists():
        subprocess.run(['git', 'clone', url, str(path)], check=True)
    if subprocess.check_output(['git', '-C', str(path), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError(f'Checkout has local changes: {path}')
    subprocess.run(['git', '-C', str(path), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(path), 'checkout', '--detach', revision], check=True)
    assert subprocess.check_output(['git', '-C', str(path), 'rev-parse', 'HEAD'], text=True).strip() == revision

if importlib.util.find_spec('openai') is None or importlib.util.find_spec('tqdm') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai>=1.50,<3', 'tqdm'], check=True)
sys.path.insert(0, str(benchmark / 'scripts'))
sys.path.insert(0, str(project / 'src'))
from run_eval import parse_tool_calls
from uqroute_tc.inference.pilot import token_evidence
from uqroute_tc.parsing.canonical import parse_prediction
from uqroute_tc.uncertainty.alignment import meaningful_token_score

split_path = project / 'docs/splits/group_split_v1.json'
split = json.loads(split_path.read_text(encoding='utf-8'))
assert split['benchmark_revision'] == benchmark_revision
assert hashlib.sha256(split_path.read_bytes()).hexdigest() == manifest['split_sha256']
print('Saved development cases:', len(case_ids))
print('Source code:', code_revision)


## Per-case token and call alignment


In [ ]:
entries = []
for case_id in case_ids:
    record_path = output_dir / 'records' / (hashlib.sha256(case_id.encode()).hexdigest() + '.json')
    row = json.loads(record_path.read_text(encoding='utf-8'))
    benchmark_name = row['benchmark']
    assert row['id'] == case_id and benchmark_name in {'apibank', 'toolalpaca'}
    assert split['assignments'][case_id] == {
        'benchmark': benchmark_name, 'population': 'primary', 'split': 'development'
    }, f'Case outside the selected development population: {case_id}'
    raw = row['prediction']['raw_output']
    parsed = parse_prediction(raw, benchmark_name, parse_tool_calls)
    saved_calls = row['prediction']['tool_calls']
    parser_matches_saved = list(parsed.calls) == saved_calls
    response = row['uqroute']['response']
    choice = response['choices'][0] if response and response.get('choices') else None
    evidence = token_evidence(choice) if choice else {'valid': False, 'reason': 'missing response'}
    entry = {
        'benchmark': benchmark_name, 'case_id': case_id,
        'record_status': row['uqroute']['status'], 'parser_status': parsed.status,
        'parsed_call_count': len(parsed.calls), 'parser_matches_saved': parser_matches_saved,
        'token_evidence_valid': evidence['valid'],
        'meaningful_token_status': 'unavailable', 'reason': None,
    }
    if not parser_matches_saved:
        entry['reason'] = 'saved calls disagree with pinned parser'
    elif entry['record_status'] != 'complete' or choice is None or choice.get('finish_reason') == 'length':
        entry['reason'] = f"incomplete response: {entry['record_status']}"
    elif not evidence['valid'] or choice['message'].get('content') != raw:
        entry['reason'] = evidence.get('reason') or 'visible response differs from saved prediction'
    elif parsed.status != 'calls':
        entry['reason'] = f'no trusted parsed call: {parsed.status}'
    else:
        try:
            score = meaningful_token_score(
                raw, saved_calls, choice['logprobs']['content'],
                choice['finish_reason'], benchmark_name
            )
        except ValueError as exc:
            entry['reason'] = str(exc)
        else:
            entry.update(
                meaningful_token_status='valid',
                selected_token_count=len(score.selected_token_indices),
                boundary_crossing_token_count=len(score.boundary_crossing_token_indices),
                mean_token_nll=score.mean_token_nll,
            )
    entries.append(entry)

assert len(entries) == 24
assert Counter(e['benchmark'] for e in entries) == {'apibank': 12, 'toolalpaca': 12}
print('Processed:', len(entries))
print('Parser agreement:', sum(e['parser_matches_saved'] for e in entries))
print('Valid token evidence:', sum(e['token_evidence_valid'] for e in entries))


## Compact audit


In [ ]:
summary = {}
for name in ('apibank', 'toolalpaca'):
    subset = [entry for entry in entries if entry['benchmark'] == name]
    summary[name] = {
        'cases': len(subset),
        'parser_status_counts': dict(Counter(e['parser_status'] for e in subset)),
        'meaningful_token_status_counts': dict(Counter(e['meaningful_token_status'] for e in subset)),
        'unavailable_reasons': dict(Counter(e['reason'] for e in subset if e['reason'])),
    }
    print(name, '| eligible:', summary[name]['meaningful_token_status_counts'].get('valid', 0),
          '/', len(subset), '| parser:', summary[name]['parser_status_counts'])
    print('  unavailable:', summary[name]['unavailable_reasons'])

report = {
    'scope': '24 saved clean development APIBank and ToolAlpaca cases; no inference',
    'source_manifest_sha256': hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    'benchmark_revision': benchmark_revision,
    'split_sha256': manifest['split_sha256'],
    'source_code_revision': manifest['code_revision'],
    'audit_code_revision': code_revision,
    'model_id': manifest['model_id'],
    'model_revision': manifest['model_revision'],
    'case_count': len(entries), 'summary': summary, 'entries': entries,
    'interpretation_limit': 'Development format eligibility only. Parser status and meaningful-token availability do not measure official correctness; unavailable scores are not imputed.',
}
audit_path = output_dir / 'alignment_coverage_audit_v1.json'
audit_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Saved:', audit_path)
files.download(str(audit_path))
